# 03 — Ablations & Inference

Summary of the 9-experiment hyperparameter sweep and interactive inference examples.

In [ ]:
import json
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

with open('../results/metrics.json') as f:
    metrics = json.load(f)

df = pd.DataFrame(metrics['ablations'])
df

In [ ]:
# Key finding: MAX_LENGTH=512 vs 256
df_test = df.dropna(subset=['test_acc'])

fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ['steelblue' if m == 256 else 'tomato' for m in df_test['max_len']]
ax.bar(df_test['exp'].astype(str), df_test['test_acc'], color=colors)
ax.axhline(0.92, linestyle='--', color='green', label='target 92%')
ax.set_xlabel('Experiment'); ax.set_ylabel('Test Accuracy')
ax.set_title('Ablation results — red=max_len 512, blue=max_len 256')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/ablations.png', dpi=120)
plt.show()

In [ ]:
print(f"Best test accuracy: {metrics['test_acc']*100:.2f}%")
print(f"Best val  accuracy: {metrics['val_acc']*100:.2f}%")

## Inference examples

In [ ]:
from src.utils import load_model, predict, predict_batch

model, tokenizer = load_model(
    checkpoint_path='../checkpoints/best.pt',
    model_checkpoint='distilbert-base-uncased-finetuned-sst-2-english',
)
print('Model loaded')

In [ ]:
text = 'This movie was absolutely amazing!'
label, conf = predict(text, model, tokenizer)
print(f'{label} ({conf:.2%}) — {text}')

In [ ]:
reviews = [
    'Terrible waste of time',
    "Best film I've seen this year",
    'Not bad, but could be better',
]
results = predict_batch(reviews, model, tokenizer)
for text, (label, conf) in zip(reviews, results):
    print(f'{label:8} ({conf:.2%}) — {text}')